# Урок 12. Интерпретация бустинга

**Интерактивная версия:** `lessons/lesson_12/web/index.html`

1. Разложение прогноза на вклады: φ₀ + Σφ = F(x).
2. Наши точные значения Шепли против библиотеки `shap` (TreeExplainer) на модели XGBoost.
3. Глобальная важность как средний |φ|.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "python" / "gbcourse").is_dir())
sys.path.insert(0, str(ROOT / "shared" / "python"))

import numpy as np
import pandas as pd
import shap
import xgboost as xgb
from gbcourse import datasets, GBRegressor
from gbcourse.explain import shapley_values

X, y = datasets.friedman1(n=600, noise=1.0, seed=180, n_features=7)
model = GBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1).fit(X, y)

## 1. База + сумма вкладов = прогноз

In [2]:
rows = []
for i in range(5):
    base, phi = shapley_values(model, X[i])
    rows.append({"объект": i, "φ₀": base, **{f"φ(x{j})": phi[j] for j in range(7)},
                 "φ₀ + Σφ": base + phi.sum(), "F(x)": model.predict(X[i:i + 1])[0]})
pd.DataFrame(rows).round(3)

,объект,φ₀,φ(x0),φ(x1),φ(x2),φ(x3),φ(x4),φ(x5),φ(x6),φ₀ + Σφ,F(x)
0,0,14.449,1.525,1.868,-1.071,1.275,0.165,0.092,-0.035,18.269,18.269
1,1,14.449,0.498,2.207,-1.371,-3.801,1.648,-0.200,-0.055,13.375,13.375
2,2,14.449,-3.862,1.189,0.644,-2.254,2.119,0.061,-0.034,12.313,12.313
3,3,14.449,1.120,-4.153,-0.717,2.007,-2.375,0.189,0.003,10.523,10.523
4,4,14.449,0.853,2.233,-0.110,1.390,1.822,0.071,-0.046,20.662,20.662


## 2. Сверка с shap.TreeExplainer

Обучим XGBoost и посмотрим, как ту же задачу решает библиотека `shap`: TreeSHAP считает точные значения Шепли
за полиномиальное время, и для них тоже выполняется аддитивность.

In [3]:
xm = xgb.XGBRegressor(n_estimators=50, max_depth=3, learning_rate=0.1, base_score=float(y.mean())).fit(X, y)
explainer = shap.TreeExplainer(xm)
sv = explainer.shap_values(X[:20])
print("shap: база", float(np.ravel(explainer.expected_value)[0]))
print("аддитивность shap: наибольшая |φ₀ + Σφ − F(x)| =",
      np.abs(explainer.expected_value + sv.sum(1) - xm.predict(X[:20], output_margin=True)).max())
print("вклады первого объекта:", np.round(sv[0], 3))

shap: база 14.451026916503906
аддитивность shap: наибольшая |φ₀ + Σφ − F(x)| = 5.722046e-06
вклады первого объекта: [ 1.318  1.629 -0.878  1.208 -0.056  0.034  0.   ]


Сверку «наш перебор = shap до 1e-15» на моделях scikit-learn выполняет тест `tests/python/test_parity.py`.

## 3. Глобальная важность — средний |φ|

In [4]:
phis = np.array([shapley_values(model, X[i])[1] for i in range(200)])
imp = pd.Series(np.abs(phis).mean(0), index=[f"x{j}" for j in range(7)]).sort_values(ascending=False)
imp.round(3)

x3    2.368
x1    1.778
x0    1.498
x4    1.178
x2    1.084
x5    0.117
x6    0.064
dtype: float64

## Упражнения

1. Найдите объект с наибольшим |F(x) − φ₀| и опишите, какие признаки «сделали» его прогноз.
2. Сравните средний |φ| со встроенной важностью `model.feature_importances_` (доля суммарного выигрыша). Совпадает ли порядок?

Решения: `python lessons/lesson_12/exercises/solutions.py`.